# RealPDE Baselines — sim↔real matrix

Run **all 8 shipped baselines** against **all 3 directions** using the Kaggle Dataset mounted at `/kaggle/input/datasets/nthday/realpde/`. **Real eval uses `train_real/` only (`test/` is ignored). Each setting loads a deterministic 40% of `.h5` files** (sorted tail, staged once per split) and scores all their windows.

| # | Direction | Checkpoints | Data | Meaning |
|---|---|---|---|---|
| A | **sim → sim** | `baseline/sim_pretrain/*` | `train_sim/` (val holdout) | in-distribution ceiling |
| B | **sim → real (zero-shot)** | `baseline/sim_pretrain/*` | `train_real/` (40% of files) | sim-to-real gap, no adaptation |
| C | **sim+real → real (finetuned)** | `baseline/sim_real_ft/*` | `train_real/` (40% of files) | best supervised reference |
| D | **real → sim (reverse)** | `baseline/sim_real_ft/*` | `train_sim/` (40% of files) | forgetting check: what finetuning cost on sim |

Method: **direct (non-TTT) teacher-forcing eval** in raw space — `PDEDataset(in=20, out=20, interval=20, sub_s=2)` over a staged 40% file subset + `load_baseline`, reporting **MSE** and **rel-L2** (same formula as `scoring.py`, unnormalized). File-level sampling keeps the disk read at 40%; the per-split stage is cached and reused across settings.

**Repo is public** — no `GITHUB_TOKEN` needed. Reference pattern: `notebook/pretrain_kaggle.ipynb`.

In [ ]:
# --- Clone or pull repo (public, no token) ---
REPO_DIR = '/kaggle/working/realpde'
REPO_URL = 'https://github.com/nthday-jpg/realpde.git'

!if [ -d {REPO_DIR} ]; then echo "Pulling {REPO_DIR}..."; cd {REPO_DIR} && git pull; else echo "Cloning {REPO_URL}..."; git clone {REPO_URL} {REPO_DIR}; fi

In [ ]:
%cd {REPO_DIR}

In [ ]:
# Enforce Python 3.10 (starterkit) — .python-version pins it
!uv python pin 3.10 2>&1 | tail -1
!uv sync --python 3.10

In [ ]:
# Install into the KERNEL interpreter, not uv's managed Python.
# Kernel here is 3.12 but pyproject pins ==3.10.* -> plain install refuses,
# so force it: package is pure-Python, safe on 3.12. Falls back to src/ on
# sys.path if the install ever fails (helper cell also does that).
import sys, os
_REPO = os.environ.get('REPO_DIR', '/kaggle/working/realpde')
!{sys.executable} -m pip install -q -e . --no-deps --ignore-requires-python || echo install-failed-falling-back-to-src
for _p in (_REPO + '/src', _REPO):
    sys.path.insert(0, _p) if _p not in sys.path else None
import realpde; print('realpde OK ->', realpde.__file__)


In [ ]:
# --- GPU check ---
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Count: {torch.cuda.device_count()}')

In [ ]:
# =============================================================================
# CONFIG — edit these before running
# =============================================================================
import os
from pathlib import Path

# Kaggle Dataset root from Image 1 (right panel):
#   /kaggle/input/datasets/nthday/realpde/{baseline,test,train_real,train_sim}/
# Fallbacks cover: renamed slug (/kaggle/input/realpde) and local ./data layout.
ROOT_CANDIDATES = [
    '/kaggle/input/datasets/nthday/realpde',
    '/kaggle/input/realpde',
    './data',
    'data',
]
DATA_ROOT = next((c for c in ROOT_CANDIDATES if Path(c).exists()), ROOT_CANDIDATES[0])
os.environ['DATA_ROOT'] = DATA_ROOT

# Eval knobs
os.environ.setdefault('IN_STEP', '20')
os.environ.setdefault('OUT_STEP', '20')
os.environ.setdefault('INTERVAL', '20')
os.environ.setdefault('SUB_S', '2')
BATCH_SIZE = int(os.environ.get('BASELINE_BATCH_SIZE', '4'))  # FNO-fp32 is 400MB; keep small
SAMPLE_FRAC = float(os.environ.get('BASELINE_SAMPLE_FRAC', '0.4'))  # 40% of .h5 FILES per split (file-level: faster load)
# Prefer fp16-packed FNO (192MB) over fp32 (384MB) on 16GB Kaggle GPUs:
PREFER_FP16_FNO = os.environ.get('PREFER_FP16_FNO', '1') == '1'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f'DATA_ROOT -> {DATA_ROOT}')
print(f'DEVICE={DEVICE} BATCH_SIZE={BATCH_SIZE} SAMPLE_FRAC={SAMPLE_FRAC} PREFER_FP16_FNO={PREFER_FP16_FNO}')

In [ ]:
# --- Probe dataset layout (Image 1) + resolve h5 dirs ---
from pathlib import Path
import os

root = Path(os.environ['DATA_ROOT'])
print(f'root: {root} exists={root.exists()}')
if root.exists():
    for child in sorted(root.iterdir()):
        n_h5 = len(list(child.rglob('*.h5'))) if child.is_dir() else 0
        print(f'  {child.name:12s} {"dir" if child.is_dir() else "file":4s}  ({n_h5} .h5 underneath)')

def resolve_h5_dir(*names):
    """Return the deepest dir under DATA_ROOT/<names...> that directly holds .h5 files.
    Handles flat (train_sim/*.h5) and nested (train_sim/train_sim/*.h5) layouts."""
    for name in names:
        base = root / name
        if not base.exists():
            continue
        direct = sorted(base.glob('*.h5'))
        if direct:
            return base
        nested = sorted([p for p in base.rglob('*.h5')])
        if nested:
            # most common parent dir holding .h5 files
            from collections import Counter
            parent, _ = Counter(p.parent for p in nested).most_common(1)[0]
            return parent
    return None

SIM_DIR  = resolve_h5_dir('train_sim')
REAL_TR  = resolve_h5_dir('train_real')  # NOTE: test/ intentionally ignored — real eval uses train_real/ only
print('test/ ignored by design (train_real is the real split)')

for label, d in [('train_sim (SIM)', SIM_DIR), ('train_real (REAL)', REAL_TR)]:
    if d is None:
        print(f'[missing] {label}')
    else:
        files = sorted(d.glob('*.h5'))
        print(f'{label:24s} -> {d}  ({len(files)} .h5, e.g. {files[0].name if files else "-"})')

# Baseline checkpoints (Image 1 left panel)
ckpt_files = sorted((root / 'baseline').rglob('*.pt*')) if (root / 'baseline').exists() else []
if not ckpt_files and Path('data/baseline_checkpoints').exists():
    ckpt_files = sorted(Path('data/baseline_checkpoints').rglob('*.pt*'))
print(f'\n{len(ckpt_files)} baseline checkpoint(s):')
for c in ckpt_files:
    try:
        print(f'  {c.relative_to(root) if str(c).startswith(str(root)) else c}  {c.stat().st_size/1024/1024:.0f} MB')
    except Exception:
        print(f'  {c}')

SIM_PRE = sorted([p for p in ckpt_files if 'sim_pretrain' in str(p).replace(chr(92), '/') or ('/sim_' in str(p) and 'sim_real' not in str(p).lower())])
FT      = sorted([p for p in ckpt_files if 'sim_real' in str(p).lower()])
# fallback: split by filename when folder names differ
if not SIM_PRE:
    SIM_PRE = sorted([p for p in ckpt_files if 'sim_real' not in p.name.lower()])
print(f'\nsim_pretrain: {[p.name for p in SIM_PRE]}')
print(f'sim_real_ft : {[p.name for p in FT]}')

In [ ]:
# --- Unified direct-eval helper (run once) ---
# Loads any baseline via load_baseline (handles fp32 train-ckpt + fp16-packed),
# scores PDEDataset windows: MSE + rel-L2 in RAW space (same formula as scoring.py).
# One model at a time + empty_cache so the 400MB FNO-fp32 fits on Kaggle GPUs.
# Single GPU: plain scoring. Multi-GPU: NCCL DDP subprocesses (one per GPU),
# each scoring its own file shard. Subprocesses (not DP) so both cards really work.
import gc, torch, torch.nn.functional as F
from pathlib import Path
from torch.utils.data import DataLoader, Subset
import sys
try:
    from tqdm.auto import tqdm as _tqdm  # progress bars (Kaggle has tqdm preinstalled)
except Exception:
    def _tqdm(x, **kw): return x  # fallback: no-op if tqdm missing
# src-layout project: package lives in <repo>/src/realpde, NOT <repo>/realpde —
# so <repo> alone on sys.path is NOT enough (and `uv pip --system` may target
# a different interpreter than the kernel). Put <repo>/src first.
REPO = '/kaggle/working/realpde'
import os as _os
REPO = _os.environ.get('REPO_DIR', REPO)
for _p in (f'{REPO}/src', REPO, '.', 'src'):
    if _p not in sys.path:
        sys.path.insert(0, _p)
import realpde as _rp
print(f'realpde -> {_rp.__file__}')

from realpde.datasets import PDEDataset
# h5py serializes threaded reads behind its global lock, so PDEDataset's
# ThreadPoolExecutor never scales (that is your 10% CPU): run preload in PROCESSES.
# _load_one_file is module-level/picklable, so swapping the executor class just works.
# Tune with PDE_PRELOAD_WORKERS (default cpu_count); with 2 DDP ranks, 2-4 each is plenty.
from concurrent.futures import ProcessPoolExecutor as _PPE
import realpde.datasets.pde_dataset as _pdemod
_pdemod.ThreadPoolExecutor = _PPE
print('preload backend: processes (PDE_PRELOAD_WORKERS=' + _os.environ.get('PDE_PRELOAD_WORKERS', 'cpu_count') + ')')
from load_baseline import load_baseline, detect_model_type

IN_STEP, OUT_STEP, INTERVAL, SUB_S = (int(__import__('os').environ.get(k, v)) for k, v in [('IN_STEP','20'),('OUT_STEP','20'),('INTERVAL','20'),('SUB_S','2')])
# NUM_WORKERS: DataLoader loader processes. NOTE -1 is NOT valid here
# (DataLoader requires >= 0; 0 = load in the main process). And if your GPUs
# already sit at 100%, you are compute-bound: more workers will NOT speed up
# scoring, they only add CPU contention. Raise this when GPUs starve (wait on data).
NUM_WORKERS = int(_os.environ.get('NUM_WORKERS', '2'))
# SHUFFLE_FILES: shuffle file list (seeded) before taking the 40% file subset,
# so the sample is not biased to one end of a sorted-by-condition file list.
SHUFFLE_FILES = _os.environ.get('SHUFFLE_FILES', '1') == '1'
FILE_SEED = int(_os.environ.get('FILE_SEED', '42'))

# --- Dataset cache: PDEDataset preloads whole .h5 files into RAM, so sample
# 40% at the FILE level (seeded-shuffled 40% of files, staged once as symlinks
# under /tmp/realpde_sub/<split>/). A+D share the SIM stage, B+C the REAL
# stage: 2 partial loads total instead of ~12 full ones. Disable with
# CACHE_DATASETS=0; free RAM with clear_ds_cache().
_DS_CACHE = {}
CACHE_DATASETS = _os.environ.get('CACHE_DATASETS', '1') == '1'
def _select_files(data_dir, file_frac=0.4):
    data_dir = Path(data_dir)
    files = sorted(data_dir.glob('*.h5'))
    if SHUFFLE_FILES:  # shuffle (seeded) BEFORE sampling so the 40% is unbiased
        import random as _random
        files = list(files); _random.Random(FILE_SEED).shuffle(files)
    nf = max(1, int(len(files) * file_frac))
    return files, files[:nf]
def _get_ds(data_dir, file_frac=0.4):
    import shutil as _shutil
    import time as _time
    files, sel = _select_files(data_dir, file_frac)
    data_dir = Path(data_dir)
    key = (str(data_dir), file_frac, SHUFFLE_FILES, FILE_SEED, IN_STEP, OUT_STEP, INTERVAL, SUB_S)
    if CACHE_DATASETS and key in _DS_CACHE:
        print(f'[data] reuse cached {data_dir.name}: {len(_DS_CACHE[key])} samples from {len(sel)} files (no reload)')
        return _DS_CACHE[key]
    stage = Path('/tmp/realpde_sub') / data_dir.name
    want = sorted([f.name for f in sel])
    have = sorted([p.name for p in stage.glob('*.h5')]) if stage.exists() else []
    if have != want:  # (re)stage symlinks only when the selection changed
        if stage.exists():
            _shutil.rmtree(stage)
        stage.mkdir(parents=True)
        for f in sel:
            (stage / f.name).symlink_to(f)
    t0 = _time.time()
    print(f'[data] loading {len(sel)}/{len(files)} files from {data_dir.name} (one-time cost) ...')
    ds = PDEDataset(stage, in_step=IN_STEP, out_step=OUT_STEP, interval=INTERVAL, sub_s=SUB_S)
    print(f'[data] loaded {len(ds)} samples in {_time.time()-t0:.0f}s')
    if CACHE_DATASETS:
        _DS_CACHE[key] = ds
    return ds
def clear_ds_cache():
    _DS_CACHE.clear(); gc.collect()
    print('dataset cache cleared')

def score_ckpt_on_dir(ckpt_path, data_dir, batch_size=4, frac=0.4, device='cuda'):
    ckpt_path, data_dir = Path(ckpt_path), Path(data_dir)
    mtype = detect_model_type(str(ckpt_path))
    _n_gpu = torch.cuda.device_count() if device.startswith('cuda') else 0
    if _n_gpu > 1:  # DDP: one NCCL subprocess per GPU, each scores its own file shard
        print(f'[ddp] {ckpt_path.name}: {batch_size}/GPU x {_n_gpu} GPUs')
        return _score_ddp(ckpt_path, data_dir, batch_size, frac, device, mtype)
    ds = _get_ds(data_dir, file_frac=frac)  # 40% of FILES staged once, shared after
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=NUM_WORKERS, pin_memory=device.startswith('cuda'))  # full staged set
    model, meta = load_baseline(mtype, str(ckpt_path), device=device)
    model.eval()
    mse_sum, rel_sum, cnt = 0.0, 0.0, 0
    with torch.no_grad():
        for inp, tgt in _tqdm(loader, desc=ckpt_path.name, leave=False):
            inp, tgt = inp.to(device), tgt.to(device)
            pred = model(inp)
            mse_sum += F.mse_loss(pred, tgt, reduction='sum').item()
            b = pred.shape[0]
            p, t = pred.reshape(b, -1), tgt.reshape(b, -1)
            rel_sum += ((p - t).norm(dim=1) / t.norm(dim=1).clamp_min(1e-8)).sum().item()
            cnt += b
    n_elem = cnt * IN_STEP * 32 * 64 * 3
    out = {'ckpt': ckpt_path.name, 'type': mtype, 'n': cnt,
           'mse': mse_sum / max(n_elem, 1), 'rel_l2': rel_sum / max(cnt, 1), 'meta': meta}
    del model; gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return out

# DDP worker: must live in a real file because spawned/subprocess workers cannot
# re-import notebook cells. One process per GPU, NCCL, each scores its file shard
# and writes partial sums to disk for the parent to aggregate.
_DDP_WORKER_SRC = '''
import json
import os
import sys
import torch
import torch.distributed as dist
import torch.nn.functional as F
from pathlib import Path
from torch.utils.data import DataLoader
from torch.nn.parallel import DistributedDataParallel as DDP

def main():
    job = json.loads(Path(os.environ['REALPDE_DDP_JOB']).read_text())
    repo = os.environ['REALPDE_REPO']
    for p in (repo + '/src', repo):
        if p not in sys.path:
            sys.path.insert(0, p)
    rank = int(os.environ['RANK'])
    world = int(os.environ['WORLD_SIZE'])
    torch.cuda.set_device(rank)
    dist.init_process_group('nccl', rank=rank, world_size=world)
    from realpde.datasets import PDEDataset
    from concurrent.futures import ProcessPoolExecutor as _PPE
    import realpde.datasets.pde_dataset as _pdemod
    _pdemod.ThreadPoolExecutor = _PPE
    from load_baseline import load_baseline, detect_model_type
    stage = Path(job['stage']) / ('rank' + str(rank))
    if stage.exists():
        import shutil
        shutil.rmtree(stage)
    stage.mkdir(parents=True)
    for f in job['files'][rank::world]:
        (stage / Path(f).name).symlink_to(f)
    ds = PDEDataset(stage, in_step=job['in_step'], out_step=job['out_step'], interval=job['interval'], sub_s=job['sub_s'])
    print('rank ' + str(rank) + ': ' + str(len(ds)) + ' samples', flush=True)
    loader = DataLoader(ds, batch_size=job['batch'], shuffle=False, num_workers=job['workers'])
    model, _meta = load_baseline(detect_model_type(job['ckpt']), job['ckpt'], device='cuda:' + str(rank))
    model = DDP(model, device_ids=[rank])
    model.eval()
    mse_sum = 0.0
    rel_sum = 0.0
    cnt = 0
    with torch.no_grad():
        prog_path = Path(job['progress'].replace('{rank}', str(rank)))
        prog_path.write_text(json.dumps({'done': 0, 'total': len(ds)}))
        for bi, (inp, tgt) in enumerate(loader):
            inp = inp.to('cuda:' + str(rank), non_blocking=True)
            tgt = tgt.to('cuda:' + str(rank), non_blocking=True)
            pred = model(inp)
            mse_sum += F.mse_loss(pred, tgt, reduction='sum').item()
            b = pred.shape[0]
            p = pred.reshape(b, -1)
            t = tgt.reshape(b, -1)
            rel_sum += ((p - t).norm(dim=1) / t.norm(dim=1).clamp_min(1e-8)).sum().item()
            cnt += b
            if bi % 10 == 0:
                prog_path.write_text(json.dumps({'done': cnt, 'total': len(ds)}))
    prog_path.write_text(json.dumps({'done': cnt, 'total': len(ds)}))
    out = {'mse_sum': mse_sum, 'rel_sum': rel_sum, 'cnt': cnt}
    Path(job['partial'].replace('{rank}', str(rank))).write_text(json.dumps(out))
    dist.barrier()
    dist.destroy_process_group()

if __name__ == '__main__':
    main()
'''

def _score_ddp(ckpt_path, data_dir, batch_size, frac, device, mtype):
    import json as _json
    import socket as _socket
    import subprocess as _sp
    world = torch.cuda.device_count()
    stem = Path(ckpt_path).stem
    ddir = Path('/tmp/realpde_ddp')
    ddir.mkdir(parents=True, exist_ok=True)
    (ddir / 'worker.py').write_text(_DDP_WORKER_SRC)
    files, sel = _select_files(data_dir, frac)
    s = _socket.socket(); s.bind(('127.0.0.1', 0)); port = str(s.getsockname()[1]); s.close()
    batch = batch_size
    while True:  # auto-shrink batch on OOM: CNO/FNO/Transolver differ 8x in appetite
        job = {'ckpt': str(ckpt_path), 'files': [str(f) for f in sel],
               'stage': str(ddir / ('stage_' + stem)),
               'partial': str(ddir / ('partial_' + stem + '_rank{rank}.json')),
               'progress': str(ddir / ('progress_' + stem + '_rank{rank}.json')),
               'in_step': IN_STEP, 'out_step': OUT_STEP, 'interval': INTERVAL, 'sub_s': SUB_S,
               'batch': batch, 'workers': NUM_WORKERS}
        job_path = ddir / ('job_' + stem + '.json')
        job_path.write_text(_json.dumps(job))
        print(f'[ddp] {stem}: {len(sel)}/{len(files)} files over {world} ranks, batch {batch}/GPU, port {port}')
        for _stale in list(ddir.glob('progress_' + stem + '_rank*.json')) + list(ddir.glob('partial_' + stem + '_rank*.json')):
            _stale.unlink()  # drop last run's files so the live bar starts empty
        procs = []
        for rank in range(world):
            env = dict(_os.environ)
            env.update({'RANK': str(rank), 'WORLD_SIZE': str(world),
                        'MASTER_ADDR': '127.0.0.1', 'MASTER_PORT': port,
                        'NCCL_IB_DISABLE': '1',
                        'REALPDE_DDP_JOB': str(job_path), 'REALPDE_REPO': REPO})
            logf = open(ddir / ('rank_' + stem + '_' + str(rank) + '.log'), 'w')
            procs.append([_sp.Popen([sys.executable, str(ddir / 'worker.py')], env=env, stdout=logf, stderr=_sp.STDOUT), logf, rank])
        bar = _tqdm(total=1, desc='[ddp] ' + stem)  # live bar fed by worker progress files
        import time as _time
        done_r = [0]*world; tot_r = [0]*world
        try:
            while True:
                _time.sleep(2.0)
                for rank in range(world):
                    pf = ddir / ('progress_' + stem + '_rank' + str(rank) + '.json')
                    if pf.exists():
                        try:
                            prog = _json.loads(pf.read_text())
                        except Exception:
                            continue
                        tot_r[rank] = int(prog.get('total', 0))
                        done_r[rank] = int(prog.get('done', 0))
                t = sum(tot_r)
                if t > 0 and bar.total != t:
                    bar.total = t; bar.refresh()
                bar.update(sum(done_r) - bar.n)
                if all(p.poll() is not None for p, _, _ in procs):
                    break
        finally:
            bar.close()
        ok = True; oom = False
        for p, logf, rank in procs:
            try:
                rc = p.wait(timeout=7200)
            except _sp.TimeoutExpired:
                p.kill(); logf.close()
                raise RuntimeError('ddp rank ' + str(rank) + ' timed out after 2h')
            logf.close()
            if rc != 0:
                ok = False
                tail = (ddir / ('rank_' + stem + '_' + str(rank) + '.log')).read_text()[-3000:]
                if 'OutOfMemoryError' in tail or 'out of memory' in tail:
                    oom = True
                    if batch > 1:
                        print(f'[ddp] rank {rank} OOM at batch {batch}/GPU (auto-retry; full trace in rank log)')
                    else:
                        print(f'[ddp] rank {rank} FAILED (rc={rc}):\n' + tail)
                else:
                    print(f'[ddp] rank {rank} FAILED (rc={rc}):\n' + tail)
        if ok:
            break
        if oom and batch > 1:
            batch = max(1, batch // 2)
            print(f'[ddp] OOM -> retrying {stem} at batch {batch}/GPU')
            continue
        raise RuntimeError('ddp worker failed, see logs above')
    mse_sum = 0.0; rel_sum = 0.0; cnt = 0
    for rank in range(world):
        part = _json.loads((ddir / ('partial_' + stem + '_rank' + str(rank) + '.json')).read_text())
        mse_sum += part['mse_sum']; rel_sum += part['rel_sum']; cnt += part['cnt']
    n_elem = cnt * IN_STEP * 32 * 64 * 3
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return {'ckpt': Path(ckpt_path).name, 'type': mtype, 'n': cnt,
            'mse': mse_sum / max(n_elem, 1), 'rel_l2': rel_sum / max(cnt, 1),
            'meta': {'backend': 'ddp', 'world': world}}

def pick_fno(ckpts, prefer_fp16=True):
    """Avoid loading BOTH 400MB fp32 + 200MB fp16 FNO on small GPUs — pick one."""
    fno = [p for p in ckpts if 'fno' in p.name.lower()]
    if len(fno) <= 1 or not prefer_fp16:
        return ckpts
    keep_fp16 = any('fp16' in p.name.lower() for p in fno)
    drop = 'fp16' not in 'x'  # placeholder
    out = []
    for p in ckpts:
        if 'fno' in p.name.lower() and keep_fp16 and 'fp16' not in p.name.lower():
            print(f'[skip] {p.name} (fp32 duplicate — scoring fp16 twin instead; set PREFER_FP16_FNO=0 to keep)')
            continue
        out.append(p)
    return out

print(f'scorer ready: in={IN_STEP} out={OUT_STEP} interval={INTERVAL} sub_s={SUB_S} workers={NUM_WORKERS} gpus={torch.cuda.device_count()} shuffle_files={SHUFFLE_FILES} seed={FILE_SEED}')

## A — sim → sim (in-distribution ceiling)

`sim_pretrain/*` on 40% of `train_sim/` files. Expect the lowest rel-L2 in the whole notebook.

In [ ]:
# --- A: sim -> sim ---
import pandas as pd
results = {}
assert SIM_DIR is not None, 'train_sim/ not found under DATA_ROOT — check dataset attachment'
for ckpt in _tqdm(pick_fno(SIM_PRE, PREFER_FP16_FNO), desc='checkpoints'):
    print(f'[A sim->sim] {ckpt.name} on {SIM_DIR} ...')
    try:
        r = score_ckpt_on_dir(ckpt, SIM_DIR, batch_size=BATCH_SIZE, frac=SAMPLE_FRAC, device=DEVICE)
        print(f"    MSE {r['mse']:.6f}  rel-L2 {r['rel_l2']:.6f}  (n={r['n']})")
        results[('A sim->sim', ckpt.name)] = r
    except Exception as e:
        err = f'{type(e).__name__}: {e}'
        print(f'    FAILED: {err[:160]}')
        results[('A sim->sim', ckpt.name)] = {'ckpt': ckpt.name, 'type': '?', 'n': 0, 'mse': None, 'rel_l2': None, 'error': err[:300]}
pd.DataFrame([{'setting': k[0], 'ckpt': k[1], 'mse': v['mse'], 'rel_l2': v['rel_l2'], 'n': v['n']} for k, v in results.items() if k[0].startswith('A')])

## B — sim → real, zero-shot (the sim-to-real gap)

Same `sim_pretrain/*` checkpoints, now on real data (40% of `train_real/` files — `test/` is not used). The A→B delta **is** the sim-to-real gap.

In [ ]:
# --- B: sim -> real (zero-shot, train_real only, 40%) ---
REAL_EVAL_DIR = REAL_TR
assert REAL_EVAL_DIR is not None, 'neither test/ nor train_real/ found under DATA_ROOT'
print(f'[B] real eval dir -> {REAL_EVAL_DIR}')
for ckpt in _tqdm(pick_fno(SIM_PRE, PREFER_FP16_FNO), desc='checkpoints'):
    print(f'[B sim->real] {ckpt.name} on {REAL_EVAL_DIR} ...')
    try:
        r = score_ckpt_on_dir(ckpt, REAL_EVAL_DIR, batch_size=BATCH_SIZE, frac=SAMPLE_FRAC, device=DEVICE)
        print(f"    MSE {r['mse']:.6f}  rel-L2 {r['rel_l2']:.6f}  (n={r['n']})")
        results[('B sim->real', ckpt.name)] = r
    except Exception as e:
        err = f'{type(e).__name__}: {e}'
        print(f'    FAILED: {err[:160]}')
        results[('B sim->real', ckpt.name)] = {'ckpt': ckpt.name, 'type': '?', 'n': 0, 'mse': None, 'rel_l2': None, 'error': err[:300]}
pd.DataFrame([{'setting': k[0], 'ckpt': k[1], 'mse': v['mse'], 'rel_l2': v['rel_l2'], 'n': v['n']} for k, v in results.items() if k[0].startswith('B')])

## C — sim+real → real (finetuned reference)

`sim_real_ft/*` on the same real dir as B. C should beat B on every architecture — that margin is the value of real-data finetuning.

In [ ]:
# --- C: sim+real -> real (finetuned) ---
assert REAL_EVAL_DIR is not None
for ckpt in _tqdm(pick_fno(FT, PREFER_FP16_FNO), desc='checkpoints'):
    print(f'[C ft->real] {ckpt.name} on {REAL_EVAL_DIR} ...')
    try:
        r = score_ckpt_on_dir(ckpt, REAL_EVAL_DIR, batch_size=BATCH_SIZE, frac=SAMPLE_FRAC, device=DEVICE)
        print(f"    MSE {r['mse']:.6f}  rel-L2 {r['rel_l2']:.6f}  (n={r['n']})")
        results[('C ft->real', ckpt.name)] = r
    except Exception as e:
        err = f'{type(e).__name__}: {e}'
        print(f'    FAILED: {err[:160]}')
        results[('C ft->real', ckpt.name)] = {'ckpt': ckpt.name, 'type': '?', 'n': 0, 'mse': None, 'rel_l2': None, 'error': err[:300]}
pd.DataFrame([{'setting': k[0], 'ckpt': k[1], 'mse': v['mse'], 'rel_l2': v['rel_l2'], 'n': v['n']} for k, v in results.items() if k[0].startswith('C')])

## D — real → sim (reverse / forgetting check)

Finetuned `sim_real_ft/*` back on `train_sim/`. D vs A on the same sim data measures **catastrophic forgetting**: how much sim skill finetuning cost.

In [ ]:
# --- D: finetuned -> sim (reverse) ---
assert SIM_DIR is not None
for ckpt in _tqdm(pick_fno(FT, PREFER_FP16_FNO), desc='checkpoints'):
    print(f'[D ft->sim] {ckpt.name} on {SIM_DIR} ...')
    try:
        r = score_ckpt_on_dir(ckpt, SIM_DIR, batch_size=BATCH_SIZE, frac=SAMPLE_FRAC, device=DEVICE)
        print(f"    MSE {r['mse']:.6f}  rel-L2 {r['rel_l2']:.6f}  (n={r['n']})")
        results[('D ft->sim', ckpt.name)] = r
    except Exception as e:
        err = f'{type(e).__name__}: {e}'
        print(f'    FAILED: {err[:160]}')
        results[('D ft->sim', ckpt.name)] = {'ckpt': ckpt.name, 'type': '?', 'n': 0, 'mse': None, 'rel_l2': None, 'error': err[:300]}
pd.DataFrame([{'setting': k[0], 'ckpt': k[1], 'mse': v['mse'], 'rel_l2': v['rel_l2'], 'n': v['n']} for k, v in results.items() if k[0].startswith('D')])

## Summary matrix + plot

One table for the full sim↔real matrix, plus a grouped rel-L2 bar chart. Save to `/kaggle/working/baseline_matrix.csv`.

In [ ]:
# --- Summary: full sim<->real matrix ---
import pandas as pd, matplotlib.pyplot as plt
from pathlib import Path

rows = []
for k, v in results.items():
    rows.append({'setting': k[0], 'ckpt': k[1], 'arch': v.get('type', '?'),
                 'mse': v.get('mse'), 'rel_l2': v.get('rel_l2'), 'n': v.get('n', 0),
                 'status': ('FAIL: ' + v['error'][:100]) if v.get('error') else 'ok'})
df = pd.DataFrame(rows)
df = df.sort_values(['arch', 'setting']).reset_index(drop=True)
display(df)  # ONE place: every setting x ckpt, ok rows + FAIL rows with reason
fails = df[df['status'] != 'ok']
if not fails.empty:
    print(f'{len(fails)} FAILED setting(s) (full traces stay in the rank logs):')
    display(fails[['setting', 'ckpt', 'status']])

out_csv = Path('/kaggle/working/baseline_matrix.csv')
df.to_csv(out_csv, index=False)
print(f'saved -> {out_csv}')

# Grouped bar chart: rel-L2 per arch across A/B/C/D
df_ok = df[df['status'] == 'ok']  # plot + deltas on successes only
if not df_ok.empty:
    piv = df_ok.pivot(index='arch', columns='setting', values='rel_l2')
    ax = piv.plot(kind='bar', figsize=(10, 4))
    ax.set_ylabel('rel-L2 (raw space, lower is better)')
    ax.set_title(f'Baseline sim<->real matrix (40% of files per split)')
    plt.xticks(rotation=0); plt.tight_layout(); plt.show()
    # Print the three headline deltas per arch
    for arch in piv.index:
        row = piv.loc[arch]
        gap = f"{row.get('B sim->real', float('nan'))/row.get('A sim->sim', float('nan')):.2f}x" if row.get('A sim->sim') else '?'
        print(f"{arch}: sim->real gap (B/A)={gap}, ft gain (B/C)={row.get('B sim->real', float('nan'))/row.get('C ft->real', float('nan')):.2f}x" if row.get('C ft->real') else f'{arch}: no ft row')